In [ ]:
UPDATE_INTERVAL = 1.0  # Update decision every 1 second
RAMP_RATE_UP = FLOOD_WATER_LEVEL / RAMP_UP_TIME  # meters per second
RAMP_RATE_DOWN = (FLOOD_WATER_LEVEL - BASELINE_WATER_LEVEL) / RAMP_DOWN_TIME  # meters per second

print("Starting control loop...")
print(f"Update interval: {UPDATE_INTERVAL}s\n")

loop_count = 0
last_update_time = time.time()

try:
    while True:
        loop_count += 1
        current_time = time.time()
        elapsed = current_time - last_update_time
        
        with state["lock"]:
            severity = state["last_trigger_severity"]
            current_level = state["current_water_level"]
        
        # Update water level based on trigger severity
        if severity == "high":
            # Ramp up to flood level
            target = FLOOD_WATER_LEVEL
            desired_level = min(current_level + RAMP_RATE_UP * elapsed, target)
        elif severity == "low":
            # Stay at baseline
            desired_level = BASELINE_WATER_LEVEL
        else:
            # Recover gradually
            desired_level = max(current_level - RAMP_RATE_DOWN * elapsed, BASELINE_WATER_LEVEL)
        
        with state["lock"]:
            state["current_water_level"] = desired_level
        
        # Check if we crossed threshold
        should_alert = desired_level >= WATER_LEVEL_THRESHOLD
        was_alerted = state["alerted"]
        
        if should_alert and not was_alerted:
            print(f"[{loop_count}] ⚠️  ALERT: Water level {desired_level:.2f}m >= threshold {WATER_LEVEL_THRESHOLD}m")
            publish_alert_command("alert", "high")
            state["alerted"] = True
        elif not should_alert and was_alerted:
            print(f"[{loop_count}] ✓ ALL-CLEAR: Water level {desired_level:.2f}m < threshold")
            publish_alert_command("alert", "low")
            state["alerted"] = False
        else:
            print(f"[{loop_count}] Water: {desired_level:.2f}m | Trigger: {severity} | Alert: {state['alerted']}")
        
        last_update_time = current_time
        time.sleep(UPDATE_INTERVAL)

except KeyboardInterrupt:
    print("\n\nControl loop stopped by user.")
finally:
    connector.disconnect()
    print("Disconnected from MQTT broker.")

## Control Loop

Main decision loop that:
1. Checks trigger severity
2. Updates water level (ramp up/down)
3. Issues alerts/all-clear commands based on threshold

In [ ]:
def get_iso_timestamp():
    """Return current time as ISO 8601 string."""
    return datetime.now(timezone.utc).isoformat()

def publish_alert_command(action, severity):
    """Publish a control command to response agents."""
    cmd = ControlCommand(
        action=action,
        target="all_pedestrians",
        parameters={"severity": severity},
        timestamp=get_iso_timestamp()
    )
    topic = make_topic(mqtt_cfg, "control", "command")
    payload = json.dumps(cmd.to_json())
    publisher.publish_json(topic, payload, qos=1)
    print(f"  → Published control command: {action} ({severity})")
    return cmd

def on_trigger_message(client, userdata, msg):
    """Callback when trigger event received."""
    try:
        data = json.loads(msg.payload.decode())
        evt = TriggerEvent.from_json(data)
        
        with state["lock"]:
            state["last_trigger_severity"] = evt.severity
        
        print(f"[{evt.timestamp}] Trigger received: {evt.severity.upper()} {evt.event}")
    except Exception as e:
        print(f"Error parsing trigger: {e}")

# Register MQTT callbacks
connector.client.on_message = on_trigger_message
trigger_topic = make_topic(mqtt_cfg, "trigger")
connector.client.subscribe(trigger_topic)

print(f"Subscribed to: {trigger_topic}")

## Decision Logic and Callbacks

This section defines the logic for:
1. Receiving trigger events
2. Updating water level simulation
3. Issuing alert commands to response agents

In [ ]:
# Connect to MQTT
connector = MqttConnector(mqtt_cfg, client_id_suffix="control")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
else:
    print("✗ Failed to connect to MQTT broker")

publisher = MqttPublisher(connector)

## Connect to MQTT Broker

In [ ]:
import time
import json
import threading
from datetime import datetime, timezone

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher, make_topic
from simulated_city.flood import TriggerEvent, ControlCommand

# Load configuration
cfg = load_config()
mqtt_cfg = cfg.mqtt
flood_cfg = cfg.flood

# Control thresholds
WATER_LEVEL_THRESHOLD = 5.0  # meters - trigger evacuation
FLOOD_WATER_LEVEL = 6.5  # meters during high trigger
BASELINE_WATER_LEVEL = 0.2  # meters baseline
RAMP_UP_TIME = 5.0  # seconds to reach flood level
RAMP_DOWN_TIME = 10.0  # seconds to recover

# State tracking
state = {
    "last_trigger_severity": None,
    "current_water_level": BASELINE_WATER_LEVEL,
    "alerted": False,
    "lock": threading.Lock()
}

print("Control agent configured:")
print(f"  Water threshold: {WATER_LEVEL_THRESHOLD}m")
print(f"  Flood level: {FLOOD_WATER_LEVEL}m")
print(f"  Baseline level: {BASELINE_WATER_LEVEL}m")

## Setup and Configuration

# Agent: Control (Decision Logic)

**Role:** Central decision engine for the flood response system.

**Responsibilities:**
1. Subscribe to trigger events and sensor data
2. Simulate water level changes based on trigger severity
3. Monitor water level threshold (5m)
4. Issue evacuation commands when threshold exceeded
5. Coordinate with response agents

**Decision Logic:**
- LOW trigger → Water stays baseline
- HIGH trigger → Water rises to 6-7m
- Water >= 5m → Issue "alert" command with severity
- Water recovers → Issue "all-clear" command

# Agent Control
Logic notebook that subscribes to observer data and trigger events, then emits `ControlCommand` messages.